In [ ]:
pip install nvcc4jupyter

In [ ]:
%load_ext nvcc4jupyter

In [ ]:
%%writefile verify.py
import numpy as np
import os
import sys

def verify():
    script_dir = os.path.dirname(os.path.abspath(__file__))
    
    with open(os.path.join(script_dir, "begin.txt"), 'r') as f:
        lines = f.readlines()
    
    first_row = lines[1].split()
    size = len(first_row)
    
    A = []
    i = 1 
    for _ in range(size):
        A.append(list(map(int, lines[i].split())))
        i += 1
    
    i += 1
    B = []
    for _ in range(size):
        B.append(list(map(int, lines[i].split())))
        i += 1
    
    A = np.array(A)
    B = np.array(B)
    
    with open(os.path.join(script_dir, "end.txt"), 'r') as f:
        lines = f.readlines()
    
    C_prog = []
    for line in lines[1:1+size]:
        C_prog.append(list(map(int, line.split())))
    
    C_prog = np.array(C_prog)
    
    C_np = np.dot(A, B)
    
    if np.array_equal(C_np, C_prog):
        print("Правильно")
        sys.exit(0)
    else:
        print("Не правильно")
        sys.exit(1)

if __name__ == "__main__":
    verify()

In [ ]:
%%cuda
#include <chrono>
#include <fstream>
#include <cstdlib>
#include <iostream>
#include <vector>
#include <algorithm>
#include <random>      
#include <type_traits>

template <typename T>
__global__ void kernel(const T* A, const T* B, T* C, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    
    if (row < N && col < N) {
        T sum = 0;
        for (int k = 0; k < N; k++) {
            sum += A[row * N + k] * B[k * N + col];
        }
        C[row * N + col] = sum;
    }
}



template <typename T>
class Matrix{
    std::vector<T> arr;
    size_t N;
    
    public:
    Matrix() : N(0), arr() {}
    
    Matrix(size_t size) : N(size), arr(size * size) {}
    
    Matrix(const T& min, const T& max, const unsigned int seed, size_t size) : N(size), arr(size * size) {
        std::mt19937 gen(seed);
        std::uniform_int_distribution<T> dist(min, max);
        for(size_t i = 0; i < N*N; i++){
            arr[i] = dist(gen);
        }
    }

    size_t size() const { return N; }
    
    T* data() { return arr.data(); }
    const T* data() const { return arr.data(); }

    T& operator()(const size_t i, const size_t j){
        return arr[i*N + j];
    }

    const T& operator()(const size_t i, const size_t j) const{
        return arr[i*N + j];
    }

    Matrix<T> cuda_multi1(const Matrix<T>& m) const {
        if(N != m.N) throw "Size mismatch";
        Matrix<T> res(N);
        
        T *d_A, *d_B, *d_C;
        size_t bytes = N * N * sizeof(T);
        
        cudaMalloc(&d_A, bytes);
        cudaMalloc(&d_B, bytes);
        cudaMalloc(&d_C, bytes);
        
        cudaMemcpy(d_A, this->data(), bytes, cudaMemcpyHostToDevice);
        cudaMemcpy(d_B, m.data(), bytes, cudaMemcpyHostToDevice);
        
        dim3 threadsPerBlock(8, 8);
        dim3 numBlocks((N + 7) / 8, (N + 7) / 8);
        
        kernel<T><<<numBlocks, threadsPerBlock>>>(d_A, d_B, d_C, N);
        
        cudaDeviceSynchronize();
        cudaMemcpy(res.data(), d_C, bytes, cudaMemcpyDeviceToHost);
        
        cudaFree(d_A);
        cudaFree(d_B);
        cudaFree(d_C);
        
        return res;
    }

    Matrix<T> cuda_multi2(const Matrix<T>& m) const {
        if(N != m.N) throw "Size mismatch";
        Matrix<T> res(N);
        
        T *d_A, *d_B, *d_C;
        size_t bytes = N * N * sizeof(T);
        
        cudaMalloc(&d_A, bytes);
        cudaMalloc(&d_B, bytes);
        cudaMalloc(&d_C, bytes);
        
        cudaMemcpy(d_A, this->data(), bytes, cudaMemcpyHostToDevice);
        cudaMemcpy(d_B, m.data(), bytes, cudaMemcpyHostToDevice);
        
        dim3 threadsPerBlock(16, 16);
        dim3 numBlocks((N + 15) / 16, (N + 15) / 16);
        
        kernel<T><<<numBlocks, threadsPerBlock>>>(d_A, d_B, d_C, N);
        
        cudaDeviceSynchronize();
        cudaMemcpy(res.data(), d_C, bytes, cudaMemcpyDeviceToHost);
        
        cudaFree(d_A);
        cudaFree(d_B);
        cudaFree(d_C);
        
        return res;
    }

    Matrix<T> cuda_multi3(const Matrix<T>& m) const {
        if(N != m.N) throw "Size mismatch";
        Matrix<T> res(N);
        
        T *d_A, *d_B, *d_C;
        size_t bytes = N * N * sizeof(T);
        
        cudaMalloc(&d_A, bytes);
        cudaMalloc(&d_B, bytes);
        cudaMalloc(&d_C, bytes);
        
        cudaMemcpy(d_A, this->data(), bytes, cudaMemcpyHostToDevice);
        cudaMemcpy(d_B, m.data(), bytes, cudaMemcpyHostToDevice);
        
        dim3 threadsPerBlock(32, 32);
        dim3 numBlocks((N + 31) / 32, (N + 31) / 32);
        
        kernel<T><<<numBlocks, threadsPerBlock>>>(d_A, d_B, d_C, N);
        
        cudaDeviceSynchronize();
        cudaMemcpy(res.data(), d_C, bytes, cudaMemcpyDeviceToHost);
        
        cudaFree(d_A);
        cudaFree(d_B);
        cudaFree(d_C);
        
        return res;
    }
    
    friend std::ostream& operator<<(std::ostream& os, const Matrix<T>& m){
        for(size_t i = 0; i < m.N; i++){
            for(size_t j = 0; j < m.N; j++){
                os << m(i, j) << " ";
            }
            os << "\n";
        }
        return os;
    }
};

int main(){  
    const size_t N = 2000;
    std::cout << "Matrix size: " << N << "x" << N << std::endl;
    
    Matrix<int> m1(1, 100, 8, N);
    Matrix<int> m2(-134, 670, 45, N);
        
    std::ofstream out_begin("begin.txt");
    if (out_begin.is_open()) {
        out_begin << "Matrix A:\n" << m1 << "Matrix B:\n" << m2;
        out_begin.close();
    }
    
    auto start = std::chrono::high_resolution_clock::now();
    auto res = m1.cuda_multi2(m2);
    auto end = std::chrono::high_resolution_clock::now();
    
    auto time = std::chrono::duration_cast<std::chrono::milliseconds>(end - start);
        
    std::ofstream out("end.txt");
    if (out.is_open()) {
        out << "Result Matrix:\n" << res << "\n";
        out << "Matrix's size: " << N << "x" << N << "\n";
        out << "Time: " << time.count() << " ms" << std::flush; 
        out.close();
    }
    
    cudaDeviceProp prop;
    cudaGetDeviceProperties(&prop, 0);
    std::cout << "\nGPU Device: " << prop.name << std::endl;
    
    int result = system("python3 verify.py");
        
    if (result == 0) {
        std::cout << "\nSuccess" << std::endl;
        std::cout << "Size: " << N << "x" << N << std::endl;
        std::cout << "Time: " << time.count() << " ms" << std::endl;
    } else {
        std::cout << "\nFailure." << std::endl;
    }
    
    return 0;
}

In [ ]:
cat end.txt